# BC06 — Direction de projet (notebook de démonstration)

## De quoi parle-t-on ?

Les 5 blocs précédents produisaient du **résultat** (des données, des modèles, une API). Ce
bloc-ci parle de **méthode** : comment le projet est **cadré, dimensionné, planifié, testé et
documenté** de bout en bout.

## Une convention pour ce bloc

Le projet a été réalisé **seul, dans un cadre de formation**. Mais le bloc « direction de projet »
attend un vrai dimensionnement (équipe, budget, ROI). Tout ce qui suit est donc présenté comme une
**hypothèse de dimensionnement** : *si ce POC était mené pour un commanditaire réel, voici
l'équipe, le planning, le budget et le retour sur investissement qu'on proposerait.* Les chiffres
(effectifs, TJM, gains) sont **estimés et assumés comme tels**, pas mesurés.

## Comment suivre ce notebook

1. La **problématique métier** traduite en problématique data.
2. Le **commanditaire** et les **parties prenantes**.
3. L'**équipe projet** et la matrice **RACI**.
4. Le **rétroplanning** agile (4 itérations) + jalons + mini-Gantt.
5. La **charge et le budget** (jours-homme × TJM + infra).
6. Le **ROI** chiffré.
7. Les **risques** identifiés et leur traitement.
8. Les **tests automatisés** — exécutés en direct.
9. La **documentation**, bloc par bloc.
10. La **gouvernance des données & RGPD**.

Le détail complet est dans [`docs/gestion_projet.md`](docs/gestion_projet.md) ; ce notebook en
donne la version commentée, calcule le budget et le ROI, et exécute réellement les tests.

## 0. Préparation

In [1]:
import sys
import subprocess
from pathlib import Path

_racine = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "commun").is_dir())
_bloc = _racine / "blocs" / "bc06_gestion_projet"
print("Racine du projet :", _racine)

Racine du projet : C:\Users\rasmi\Projects\projet_rncp\oiseaux_migrateurs_npdc


## 1. La problématique métier, traduite en problématique data

Tout part d'un besoin concret. Le travail de cadrage consiste à le **reformuler** dans des termes
que des méthodes data savent traiter.

| Enjeu métier | Traduction data |
|---|---|
| Savoir *quand* les oiseaux migrateurs arrivent dans le Nord-Pas-de-Calais | **Classification binaire** : présence / absence d'une espèce, par semaine et par maille |
| Comprendre si la météo explique ces arrivées | **Corrélation** + **importance des variables** d'un modèle supervisé |
| Rendre la prévision utilisable par un non-technicien | **API** + **tableau de bord** exposant la probabilité de présence |

**Problématique retenue :** *peut-on modéliser l'arrivée des migrations à partir de variables
climatiques et de la position géographique ?* — c'est le fil rouge des 5 blocs techniques.

## 2. Commanditaire et parties prenantes *(hypothèse)*

**Commanditaire supposé :** une **association régionale de protection des oiseaux** (type LPO
Hauts-de-France), en partenariat avec un Parc naturel régional. Elle anime un réseau d'une
quarantaine de bénévoles qui réalisent des **comptages de terrain** pendant les périodes de
migration.

**Besoin exprimé :** *« On a un nombre limité de sorties bénévoles ; on aimerait les concentrer
sur les semaines et les zones où la probabilité d'observer les espèces cibles est la plus
forte. »*

| Partie prenante | Ce qu'elle attend | Implication |
|---|---|---|
| Direction de l'association (**sponsor**) | Outil fiable, coût maîtrisé, valorisable auprès des financeurs | Valide le budget et les jalons |
| Coordinateur des bénévoles | Une prévision **par semaine et par zone** pour planifier les sorties | Utilisateur principal du dashboard |
| Bénévoles observateurs (~40) | Un outil **simple**, non technique | Utilisateurs finaux ; fournissent le retour terrain |
| Prestataire / bénévole IT | Code **documenté et dockerisé**, reprenable | Maintenance après livraison |
| Référent RGPD de l'association | **Aucune donnée personnelle** traitée | Consulté au cadrage (cf. point 10) |

**Périmètre.** Dans le périmètre : 4 espèces, région Nord-Pas-de-Calais, prévision hebdomadaire,
API + dashboard. Hors périmètre : application mobile, temps réel, autres régions, autres taxons.

## 3. Équipe projet et RACI *(hypothèse)*

On propose une équipe **resserrée de 3 profils + un chef de projet à temps partiel**, sur
**~6 semaines calendaires** (le découpage fonctionnel reste en 4 itérations, cf. point 4 —
les 2 semaines supplémentaires couvrent réunions, recette et aléas).

| Rôle | Mission | Blocs portés | Charge |
|---|---|---|---|
| Chef de projet / PO | Cadrage, backlog, comités, recette, soutenance | BC06 | 0,25 ETP |
| Data Engineer | Acquisition, ETL, stockage objet, Docker | BC01, BC05 | 1,0 ETP |
| Data Scientist | EDA, modèles ML, CNN images | BC02, BC03, BC04 | 1,0 ETP |
| Dev / MLOps | API, dashboard, tests automatisés, CI | BC05, BC06 | 0,5 ETP |

**Matrice RACI** — *R* réalise, *A* approuve (responsable), *C* consulté, *I* informé :

| Livrable | Chef de projet | Data Engineer | Data Scientist | Dev / MLOps | Sponsor |
|---|:--:|:--:|:--:|:--:|:--:|
| BC01 — infrastructure & ETL | A | R | C | I | I |
| BC02 — analyse exploratoire | A | C | R | I | I |
| BC03 — machine learning | A | I | R | C | I |
| BC04 — deep learning (images) | A | I | R | C | I |
| BC05 — API / dashboard / Docker | A | C | C | R | I |
| BC06 — pilotage, tests, doc | R / A | C | C | C | C |
| Recette & soutenance | R | C | C | C | A |

## 4. Le rétroplanning (méthode agile, 4 itérations)

Le projet est découpé en **itérations d'une semaine**, chacune livrant quelque chose de
**vérifiable**. Planification **à rebours** depuis la date de soutenance.

| Itération | Bloc(s) | Livrable vérifiable | Dépend de |
|---|---|---|---|
| S1 | BC01 | Acquisition GBIF + Open-Meteo, ETL, grille présence/absence | — |
| S2 | BC02 | Saisonnalité, distributions, corrélations, test χ² | BC01 |
| S3 | BC03 + BC04 | 3 modèles ML comparés + validation + segmentation ; CNN transfer learning | BC01 |
| S4 | BC05 + BC06 | API, dashboard, Docker ; tests, documentation, cette note | BC03 |
| Soutenance | — | Support oral (10 min) + démo live | tous |

**Jalons de contrôle :**

- **J1** (fin S1) : `grille_presence_hebdo.parquet` produite → feu vert pour BC02/BC03.
- **J2** (fin S3) : modèle de production figé (`pipeline_ml.pkl`) → feu vert pour BC05.
- **J3** (mi-S4) : API + dashboard fonctionnels en local → répétition de la démo.

**Mini-Gantt** (■ = travail principal, · = appoint) :

```
                 S1     S2     S3     S4
BC01 infra      ■■■■   ·      ·      ·
BC02 EDA         ·     ■■■■   ·      ·
BC03 ML          ·      ·     ■■■■   ·
BC04 CNN         ·      ·     ■■     ·
BC05 API/Docker  ·      ·      ·     ■■■■
BC06 pilotage   ····   ····   ····   ■■■■
jalons                 J1            J2   J3
```

**Ce qui sécurise le planning.** Les fichiers `donnees/traitees/` et le modèle
`modeles/pipeline_ml.pkl` sont **versionnés dans le dépôt** : un retard sur un bloc ne bloque
pas les suivants, qui repartent de la dernière version figée. Marge : une demi-journée tampon
par itération.

## 5. Charge et budget *(hypothèse)*

**Méthode.** Pour chaque rôle : `charge (jours-homme) = ETP × durée (30 jours ouvrés)`, puis
`coût = charge × TJM`. Les TJM sont des ordres de grandeur **prestation, profils junior /
confirmé**. On ajoute l'infra de la phase projet et une contingence de 10 %.

In [2]:
# --- Hypotheses de dimensionnement (a ajuster selon le contexte reel) ---
DUREE_JOURS = 30  # ~6 semaines calendaires

equipe = {
    #  role            : (ETP,  TJM en euros)
    "Chef de projet"    : (0.25, 650),
    "Data Engineer"     : (1.0,  550),
    "Data Scientist"    : (1.0,  600),
    "Dev / MLOps"       : (0.5,  500),
}

print(f"{'Role':<16}{'j-h':>7}{'TJM':>8}{'Cout (EUR)':>13}")
print("-" * 44)
charge_totale = 0.0
cout_rh = 0.0
for role, (etp, tjm) in equipe.items():
    jh = etp * DUREE_JOURS
    cout = jh * tjm
    charge_totale += jh
    cout_rh += cout
    print(f"{role:<16}{jh:>7.1f}{tjm:>8}{cout:>13,.0f}")

infra_projet = 100          # GPU cloud ponctuel (entrainement CNN) + tests hebergement
contingence = 0.10 * cout_rh
budget_total = cout_rh + infra_projet + contingence

print("-" * 44)
print(f"{'Total j-h':<16}{charge_totale:>7.1f}")
print(f"{'Cout RH':<24}{cout_rh:>20,.0f} EUR")
print(f"{'Infra phase projet':<24}{infra_projet:>20,.0f} EUR")
print(f"{'Contingence (10%)':<24}{contingence:>20,.0f} EUR")
print(f"{'BUDGET PROJET':<24}{budget_total:>20,.0f} EUR")

Role                j-h     TJM   Cout (EUR)
--------------------------------------------
Chef de projet      7.5     650        4,875
Data Engineer      30.0     550       16,500
Data Scientist     30.0     600       18,000
Dev / MLOps        15.0     500        7,500
--------------------------------------------
Total j-h          82.5
Cout RH                               46,875 EUR
Infra phase projet                       100 EUR
Contingence (10%)                      4,688 EUR
BUDGET PROJET                         51,662 EUR


## 6. Coûts récurrents et ROI *(hypothèse)*

**Coût récurrent** après mise en production :

- Hébergement API + dashboard, stockage objet + base, domaine/TLS : **~40 €/mois**.
- Maintenance corrective et ré-entraînement : **~1 jour-homme/mois** (Dev / MLOps).

**Bénéfice attendu**, exprimé en temps bénévole **mieux employé** (et non « économisé ») :
l'outil évite des sorties de comptage à faible probabilité et redirige cet effort vers des
zones utiles. Valorisation au **barème du bénévolat** (~15 €/h). S'y ajoute un temps de
coordination évité (priorisation aujourd'hui manuelle).

In [3]:
# --- Cout recurrent annuel ---
infra_mensuelle = 40
maintenance_jh_mois = 1
tjm_maintenance = 550
cout_recurrent_an = 12 * (infra_mensuelle + maintenance_jh_mois * tjm_maintenance)

# --- Benefice annuel estime ---
nb_benevoles = 40
sorties_evitees_par_mois = 1.5      # sorties a faible proba redirigees, par benevole
mois_migration = 6                  # printemps + automne
heures_par_sortie = 3
valeur_heure_benevole = 15
temps_redeploye = (nb_benevoles * sorties_evitees_par_mois * mois_migration
                   * heures_par_sortie * valeur_heure_benevole)

coordination_evitee = 0.1 * 220 * 250   # 0,1 ETP de coordination x 220 j x cout charge/j
benefice_an = temps_redeploye + coordination_evitee

gain_net_an = benefice_an - cout_recurrent_an
retour_annees = budget_total / gain_net_an

print(f"Cout recurrent          : {cout_recurrent_an:>10,.0f} EUR/an")
print(f"  dont infra            : {12*infra_mensuelle:>10,.0f} EUR/an")
print(f"  dont maintenance      : {12*maintenance_jh_mois*tjm_maintenance:>10,.0f} EUR/an")
print(f"Benefice estime         : {benefice_an:>10,.0f} EUR/an")
print(f"  dont temps redeploye  : {temps_redeploye:>10,.0f} EUR/an")
print(f"  dont coordination     : {coordination_evitee:>10,.0f} EUR/an")
print(f"Gain net                : {gain_net_an:>10,.0f} EUR/an")
print(f"Retour sur investissement (budget / gain net) : {retour_annees:.1f} ans")

Cout recurrent          :      7,080 EUR/an
  dont infra            :        480 EUR/an
  dont maintenance      :      6,600 EUR/an
Benefice estime         :     21,700 EUR/an
  dont temps redeploye  :     16,200 EUR/an
  dont coordination     :      5,500 EUR/an
Gain net                :     14,620 EUR/an
Retour sur investissement (budget / gain net) : 3.5 ans


**Lecture.** Le retour purement financier se compte en **quelques années** : pour une
association, l'investissement se justifie donc **autant par les bénéfices qualitatifs** —
données naturalistes plus exploitables, meilleure couverture des zones à enjeu, fidélisation
des bénévoles (moins de sorties infructueuses) — que par le gain chiffré. C'est une
conclusion **assumée**, pas maquillée.

## 7. Les risques identifiés et leur traitement

Une bonne direction de projet ne **cache pas** ses limites : elle les nomme et propose des réponses.

| Risque | Prob. | Impact | Traitement | Statut |
|---|---|---|---|---|
| API GBIF indisponible (5xx transitoires) | Élevée | Moyen | `get_avec_retry` (backoff exponentiel) + `donnees/traitees/` versionnées | **Traité** |
| Fort déséquilibre des classes (~98 % d'absences) | Certaine | Élevé | Métriques adaptées (F1, AUC-ROC, matrice de confusion) plutôt que l'accuracy ; période bornée à 2019-2024 (pas d'absences fictives) ; SMOTE identifié pour la suite | **Traité (partiel)** |
| Sur-apprentissage du modèle retenu | Moyenne | Moyen | Validation croisée 5-fold + écart train/test contrôlé (< 0,05) | **Traité** |
| Météo passée seule, peu prédictive | Moyenne | Moyen | Limite assumée ; piste : intégrer des prévisions météo | **Accepté** |
| Biais d'effort d'observation (science citoyenne) | Certaine | Moyen | Signalé explicitement ; interprétation prudente | **Accepté** |
| Déploiement cloud non réalisé | Certaine | Moyen | Fichiers de déploiement prêts (`Procfile`, `render.yaml`) + procédure documentée | **Ouvert** |
| Indisponibilité d'un profil clé (Data Scientist) | Faible | Élevé | Code + notebooks pédagogiques par bloc ; artefacts figés et versionnés → reprise possible | **Traité** |
| Incompatibilité de versions au `pip install` | Faible | Faible | `requirements.txt` épinglé + `setup_venv.ps1` ; validation croisée BC03 réécrite pour être insensible aux versions | **Traité** |

## 8. Les tests automatisés : fiabiliser le code

**Pourquoi des tests ?** Un test automatisé est un petit programme qui vérifie qu'une fonction
**fait bien ce qu'elle est censée faire** sur un cas connu — par exemple : « la zone géographique
doit avoir une latitude minimale inférieure à sa latitude maximale ». C'est un **filet de
sécurité** : le jour où une modification casse quelque chose sans qu'on s'en rende compte, un test
qui échoue nous prévient.

Ici, la suite [`tests/test_acquisition.py`](tests/test_acquisition.py) porte sur le module
d'acquisition de BC01 (création de la *bounding box*, extraction des colonnes, structure des
espèces, cohérence de la zone). La cellule ci-dessous les **exécute réellement** avec `pytest`
(c'est ce que fait `run.py`, fonction `executer_tests`).

In [4]:
resultat = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--no-header"],
    cwd=str(_bloc), capture_output=True, text=True,
)
print(resultat.stdout[-3000:])
print("Tous les tests passent." if resultat.returncode == 0
      else "Certains tests ont echoue — voir le detail ci-dessus.")

============================= test session starts =============================
collected 6 items

tests\test_acquisition.py ......                                         [100%]

============================== 6 passed in 7.06s ==============================

Tous les tests passent.


**Ce que ça nous apprend.** Les 6 tests passent. Ils sont rejoués à chaque exécution de BC06 :
c'est une garantie que le code d'acquisition continue de se comporter comme prévu, même après des
modifications ultérieures (comme le passage de la période à 2019-2024).

## 9. La documentation, bloc par bloc

Chaque dossier `blocs/bc0X_.../` contient un **`README.md` au même format** : *objectif visé, ce qui
est implémenté, où le prouver dans le code, comment le démontrer en direct, statut*. Ce découpage
répété permet à n'importe quel membre du jury d'**auditer un bloc indépendamment des autres**, sans
tout relire, et de présenter chaque bloc avec sa propre commande d'exécution autonome. Chaque bloc
technique a aussi un **notebook** pédagogique.

In [5]:
for dossier in sorted((_racine / "blocs").glob("bc0*")):
    a_readme = "README" if (dossier / "README.md").exists() else "  --  "
    notebooks = [p.name for p in dossier.glob("notebook_*.ipynb")]
    a_notebook = notebooks[0] if notebooks else "(pas de notebook)"
    print(f"  {dossier.name:<34} {a_readme}   {a_notebook}")

  bc01_infrastructure_donnees        README   notebook_bc01.ipynb
  bc02_analyse_exploratoire          README   notebook_bc02.ipynb
  bc03_machine_learning              README   notebook_bc03.ipynb
  bc04_deep_learning                 README   notebook_bc04.ipynb
  bc05_industrialisation             README   notebook_bc05.ipynb
  bc06_gestion_projet                README   notebook_bc06.ipynb


## 10. Gouvernance des données & RGPD

- **RGPD** : le projet ne traite **aucune donnée à caractère personnel** (occurrences d'espèces,
  mesures météo). Détail des sources, licences et minimisation dans
  [`bc01_infrastructure_donnees/docs/architecture.md`](../bc01_infrastructure_donnees/docs/architecture.md).
- **Traçabilité** : les URL des API sont dans le code (`acquisition.py`) ; les jeux intermédiaires
  (`donnees/traitees/`) et le modèle de production (`modeles/pipeline_ml.pkl`) sont figés et
  versionnés.
- **Reproductibilité** : graines aléatoires fixées (`RANDOM_STATE`), hyperparamètres centralisés
  dans `commun/config.py`, entraînements suivis dans **MLflow** (`mlruns/`).

## Récapitulatif — ce qu'il faut retenir

BC06 ne produit pas d'algorithme : il **démontre la direction** du projet.

1. Le besoin métier a été **traduit** en un problème de classification binaire (point 1).
2. Un **commanditaire** et ses **parties prenantes** ont été identifiés (point 2).
3. Une **équipe** de 3 profils + chef de projet est proposée, avec **RACI** (point 3).
4. Le projet suit un **rétroplanning agile** de 4 itérations avec jalons J1/J2/J3 (point 4).
5. La **charge (~83 j-h)** et le **budget (~52 k€)** sont chiffrés et reproductibles (point 5).
6. Le **ROI** est calculé : retour en ~3,5 ans, complété par des bénéfices qualitatifs (point 6).
7. Les **risques** sont nommés, la plupart traités ; les limites restantes sont **assumées** (point 7).
8. Une suite de **tests automatisés** (rejoués ci-dessus, 6/6) fiabilise le code (point 8).
9. Chaque bloc est **documenté** au même format + un notebook (point 9).
10. Le projet est **hors périmètre RGPD** et **reproductible** (point 10).

> Les chiffres d'équipe, de budget et de ROI sont une **hypothèse de dimensionnement** assumée
> comme telle : le projet réel a été mené seul, en formation.

Le script [`run.py`](run.py) rejoue les tests et affiche le planning, le budget et les limites en
une commande. Le détail complet est dans [`docs/gestion_projet.md`](docs/gestion_projet.md).